# 03 — Silver: typed games + team box scores with Four Factors

Turn the raw VARIANT gamelogs into two clean, typed, deduplicated tables:

- **`team_game_box`** — one row per team per game, with the **Four Factors** computed
  (offense + defense) plus possessions and offensive/defensive ratings.
- **`games`** — one row per game (home team's perspective), with the final score and the
  `home_win` label we will model.

| Pattern | Why (Well-Architected Framework) |
|---------|----------------------------------|
| `INSERT OVERWRITE` (full refresh) | Idempotent, stateless rebuilds; MERGE isn't supported on Spark Connect (**Reliability**) |
| `MD5(...)` surrogate keys | Stable joins; both teams in a game share the same `game_sk` (**Reliability**) |
| `RELY` PK/FK + `CHECK` constraints | Optimizer hints, ERD, and enforced data-quality guards (**Reliability / Governance**) |
| `CLUSTER BY (season, game_date)` | Matches the feature/Time queries downstream (**Performance**) |

**The Four Factors** (Dean Oliver) are the box-score metrics most correlated with winning:
shooting (`eFG%`), turnovers (`TOV%`), rebounding (`ORB%`), and free throws (`FT rate`).
Each team's *defensive* four factors are simply its opponent's *offensive* four factors —
both are on the same gamelog row, so we get them for free.

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()
spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA",  "basketball_reference_waf")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"

BRONZE     = f"{UC_CATALOG}.{BRONZE_SCHEMA}.gamelog_raw"
TEAM_BOX   = f"{UC_CATALOG}.{SILVER_SCHEMA}.team_game_box"
GAMES      = f"{UC_CATALOG}.{SILVER_SCHEMA}.games"
print("Bronze source :", BRONZE)
print("Silver targets:", TEAM_BOX, "|", GAMES)

/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:141: UserWarning: Could not enable SetAllowOversizeProtos, please check the protobuf version.
  warnings.warn("Could not enable SetAllowOversizeProtos, please check the protobuf version.")


Bronze source : alexander_booth.basketball_reference_waf_bronze.gamelog_raw
Silver targets: alexander_booth.basketball_reference_waf_silver.team_game_box | alexander_booth.basketball_reference_waf_silver.games


## Constraint helpers (RELY PK/FK + CHECK)

In [2]:
def add_pk(table, name, cols):
    for c in cols:
        try:
            spark.sql(f"ALTER TABLE {table} ALTER COLUMN {c} SET NOT NULL")
        except Exception as e:
            if "ALREADY" not in str(e).upper():
                raise
    try:
        spark.sql(f"ALTER TABLE {table} DROP CONSTRAINT IF EXISTS {name}")
        spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {name} PRIMARY KEY ({', '.join(cols)}) RELY")
    except Exception as e:
        print(f"  PK {name}: {str(e)[:80]}")

def add_fk(table, name, cols, ref_table, ref_cols):
    try:
        spark.sql(f"ALTER TABLE {table} DROP CONSTRAINT IF EXISTS {name}")
        spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {name} FOREIGN KEY ({', '.join(cols)}) "
                  f"REFERENCES {ref_table} ({', '.join(ref_cols)}) RELY")
    except Exception as e:
        print(f"  FK {name}: {str(e)[:80]}")

def add_check(table, name, predicate):
    try:
        spark.sql(f"ALTER TABLE {table} DROP CONSTRAINT IF EXISTS {name}")
        spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {name} CHECK ({predicate})")
    except Exception as e:
        print(f"  CHECK {name}: {str(e)[:80]}")

print("Constraint helpers ready.")

Constraint helpers ready.


## `team_game_box` — explode, type, and compute the Four Factors

In [3]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TEAM_BOX} (
        team_game_sk STRING  NOT NULL COMMENT 'MD5(season|game_date|team) - one row per team per game',
        game_sk      STRING  NOT NULL COMMENT 'MD5(season|game_date|home_team|away_team) - shared by both teams',
        season       INT      COMMENT 'Season end year (2024-25 season => 2025)',
        game_date    DATE     COMMENT 'US/Eastern calendar date of the game',
        is_playoff   BOOLEAN  COMMENT 'TRUE for playoff games (from the gamelog playoff table)',
        team         STRING   COMMENT 'Team abbreviation (this row''s team)',
        opponent     STRING   COMMENT 'Opponent abbreviation',
        is_home      BOOLEAN  COMMENT 'TRUE if this team was at home',
        home_team    STRING,  away_team STRING,
        won          BOOLEAN  COMMENT 'Did this team win?',
        margin       INT      COMMENT 'Point margin from this team''s perspective',
        pts INT, opp_pts INT, fgm INT, fga INT, fg3m INT, fg3a INT, ftm INT, fta INT,
        orb INT, drb INT, trb INT, ast INT, stl INT, blk INT, tov INT, pf INT,
        opp_fgm INT, opp_fga INT, opp_fg3m INT, opp_fg3a INT, opp_ftm INT, opp_fta INT,
        opp_orb INT, opp_drb INT, opp_trb INT, opp_tov INT,
        efg DOUBLE         COMMENT 'Offensive effective FG%',
        tov_pct DOUBLE     COMMENT 'Offensive turnover rate',
        orb_pct DOUBLE     COMMENT 'Offensive rebound rate',
        ft_rate DOUBLE     COMMENT 'Free-throw rate (FTM/FGA)',
        def_efg DOUBLE     COMMENT 'Defensive (opponent) effective FG%',
        def_tov_pct DOUBLE COMMENT 'Defensive (opponent) turnover rate',
        drb_pct DOUBLE     COMMENT 'Defensive rebound rate',
        def_ft_rate DOUBLE COMMENT 'Defensive (opponent) free-throw rate',
        poss DOUBLE        COMMENT 'Estimated possessions',
        off_rtg DOUBLE     COMMENT 'Points per 100 possessions',
        def_rtg DOUBLE     COMMENT 'Opponent points per 100 possessions',
        net_rtg DOUBLE     COMMENT 'off_rtg - def_rtg',
        file_batch_time TIMESTAMP
    )
    CLUSTER BY (season, game_date)
    COMMENT 'Silver - one row per team per game, typed from bronze gamelogs with Four Factors.'
""")

spark.sql(f"""
    INSERT OVERWRITE {TEAM_BOX}
    WITH rows AS (
        SELECT r.value AS g, file_batch_time
        FROM {BRONZE}, LATERAL variant_explode(data) AS r
    ),
    typed AS (
        SELECT
            g:season_end_year::int  AS season,
            g:date::date            AS game_date,
            g:is_playoff::boolean   AS is_playoff,
            g:team::string          AS team,
            g:opp_name_abbr::string AS opponent,
            (g:game_location::string IS NULL OR g:game_location::string = '') AS is_home,
            g:team_game_score::int     AS pts,  g:opp_team_game_score::int AS opp_pts,
            g:fg::int   AS fgm,  g:fga::int  AS fga,
            g:fg3::int  AS fg3m, g:fg3a::int AS fg3a,
            g:ft::int   AS ftm,  g:fta::int  AS fta,
            g:orb::int  AS orb,  g:drb::int  AS drb,  g:trb::int AS trb,
            g:ast::int  AS ast,  g:stl::int  AS stl, g:blk::int AS blk,
            g:tov::int  AS tov,  g:pf::int   AS pf,
            g:opp_fg::int  AS opp_fgm,  g:opp_fga::int  AS opp_fga,
            g:opp_fg3::int AS opp_fg3m, g:opp_fg3a::int AS opp_fg3a,
            g:opp_ft::int  AS opp_ftm,  g:opp_fta::int  AS opp_fta,
            g:opp_orb::int AS opp_orb,  g:opp_drb::int  AS opp_drb,  g:opp_trb::int AS opp_trb,
            g:opp_tov::int AS opp_tov,
            file_batch_time
        FROM rows
        WHERE g:date IS NOT NULL AND g:team IS NOT NULL
    ),
    enriched AS (
        SELECT *,
            CASE WHEN is_home THEN team ELSE opponent END AS home_team,
            CASE WHEN is_home THEN opponent ELSE team END AS away_team
        FROM typed
    ),
    factors AS (
        SELECT *,
            (fgm + 0.5*fg3m) / NULLIF(fga, 0)                      AS efg,
            tov / NULLIF(fga + 0.44*fta + tov, 0)                  AS tov_pct,
            orb / NULLIF(orb + opp_drb, 0)                         AS orb_pct,
            ftm / NULLIF(fga, 0)                                   AS ft_rate,
            (opp_fgm + 0.5*opp_fg3m) / NULLIF(opp_fga, 0)         AS def_efg,
            opp_tov / NULLIF(opp_fga + 0.44*opp_fta + opp_tov, 0) AS def_tov_pct,
            drb / NULLIF(drb + opp_orb, 0)                         AS drb_pct,
            opp_ftm / NULLIF(opp_fga, 0)                           AS def_ft_rate,
            0.5*((fga + 0.44*fta - orb + tov) + (opp_fga + 0.44*opp_fta - opp_orb + opp_tov)) AS poss
        FROM enriched
    )
    SELECT
        MD5(CONCAT_WS('|', CAST(season AS STRING), CAST(game_date AS STRING), team)) AS team_game_sk,
        MD5(CONCAT_WS('|', CAST(season AS STRING), CAST(game_date AS STRING), home_team, away_team)) AS game_sk,
        season, game_date, is_playoff, team, opponent, is_home, home_team, away_team,
        (pts > opp_pts) AS won, (pts - opp_pts) AS margin,
        pts, opp_pts, fgm, fga, fg3m, fg3a, ftm, fta, orb, drb, trb, ast, stl, blk, tov, pf,
        opp_fgm, opp_fga, opp_fg3m, opp_fg3a, opp_ftm, opp_fta, opp_orb, opp_drb, opp_trb, opp_tov,
        efg, tov_pct, orb_pct, ft_rate, def_efg, def_tov_pct, drb_pct, def_ft_rate,
        poss,
        100.0*pts     / NULLIF(poss, 0) AS off_rtg,
        100.0*opp_pts / NULLIF(poss, 0) AS def_rtg,
        100.0*(pts - opp_pts) / NULLIF(poss, 0) AS net_rtg,
        file_batch_time
    FROM factors
    QUALIFY ROW_NUMBER() OVER (PARTITION BY team_game_sk ORDER BY file_batch_time DESC) = 1
""")

print("team_game_box rows:", spark.table(TEAM_BOX).count())

team_game_box rows: 7880


## `games` — one row per game (home perspective) with the label

In [4]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {GAMES} (
        game_sk      STRING  NOT NULL COMMENT 'MD5(season|game_date|home_team|away_team)',
        season       INT,
        game_date    DATE,
        is_playoff   BOOLEAN,
        home_team    STRING,
        away_team    STRING,
        home_points  INT,
        away_points  INT,
        home_win     BOOLEAN COMMENT 'Label: did the home team win?',
        home_margin  INT,
        total_points INT,
        file_batch_time TIMESTAMP
    )
    CLUSTER BY (season, game_date)
    COMMENT 'Silver - one row per game from the home team perspective; home_win is the model label.'
""")

spark.sql(f"""
    INSERT OVERWRITE {GAMES}
    SELECT
        game_sk, season, game_date, is_playoff,
        home_team, away_team,
        pts AS home_points, opp_pts AS away_points,
        won AS home_win, margin AS home_margin, (pts + opp_pts) AS total_points,
        file_batch_time
    FROM {TEAM_BOX}
    WHERE is_home = TRUE
    QUALIFY ROW_NUMBER() OVER (PARTITION BY game_sk ORDER BY file_batch_time DESC) = 1
""")

print("games rows:", spark.table(GAMES).count())

games rows: 3940


## Apply constraints (RELY PK/FK + CHECK data-quality guards)

In [5]:
add_pk(GAMES, "games_pk", ["game_sk"])
add_check(GAMES, "chk_home_points", "home_points IS NULL OR (home_points BETWEEN 30 AND 200)")
add_check(GAMES, "chk_away_points", "away_points IS NULL OR (away_points BETWEEN 30 AND 200)")

add_pk(TEAM_BOX, "team_game_box_pk", ["team_game_sk"])
add_fk(TEAM_BOX, "team_game_box_game_fk", ["game_sk"], GAMES, ["game_sk"])
add_check(TEAM_BOX, "chk_efg",  "efg IS NULL OR (efg BETWEEN 0 AND 1.2)")
add_check(TEAM_BOX, "chk_pts",  "pts IS NULL OR (pts BETWEEN 30 AND 200)")
print("Constraints applied.")

Constraints applied.


## Verify — counts, the regular/playoff split, and a Four Factors sanity check

In [6]:
spark.sql(f"""
    SELECT season,
           COUNT(*) FILTER (WHERE NOT is_playoff) AS regular_games,
           COUNT(*) FILTER (WHERE is_playoff)     AS playoff_games,
           ROUND(AVG(CASE WHEN home_win THEN 1.0 ELSE 0.0 END) FILTER (WHERE NOT is_playoff), 3) AS home_win_rate
    FROM {GAMES}
    GROUP BY season ORDER BY season
""").show()

# Four Factors should sit in sane ranges.
spark.sql(f"""
    SELECT ROUND(AVG(efg),3) AS avg_efg, ROUND(AVG(tov_pct),3) AS avg_tov_pct,
           ROUND(AVG(orb_pct),3) AS avg_orb_pct, ROUND(AVG(ft_rate),3) AS avg_ft_rate,
           ROUND(AVG(off_rtg),1) AS avg_off_rtg, ROUND(AVG(poss),1) AS avg_poss,
           MIN(efg) AS min_efg, MAX(efg) AS max_efg
    FROM {TEAM_BOX} WHERE NOT is_playoff
""").show()

+------+-------------+-------------+-------------+
|season|regular_games|playoff_games|home_win_rate|
+------+-------------+-------------+-------------+
|  2023|         1230|           84|        0.580|
|  2024|         1230|           82|        0.543|
|  2025|         1230|           84|        0.544|
+------+-------------+-------------+-------------+



+-------+-----------+-----------+-----------+-----------+--------+--------------+--------------+
|avg_efg|avg_tov_pct|avg_orb_pct|avg_ft_rate|avg_off_rtg|avg_poss|       min_efg|       max_efg|
+-------+-----------+-----------+-----------+-----------+--------+--------------+--------------+
|  0.546|      0.124|      0.243|      0.199|      112.1|   101.9|0.331325301205|0.804054054054|
+-------+-----------+-----------+-----------+-----------+--------+--------------+--------------+



Clean, typed, constrained silver tables with Four Factors. **Next:** `04_gold_star_schema`
builds the dims + facts and serving views.